In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
import pandas as pd

def extract_30_percent(input_file, output_file):
    # Load dataset
    df = pd.read_excel(input_file)

    # Assuming the label column is the last column
    label_column = df.columns[-1]

    # Extract 30% of data per label
    extracted_data = []
    for label in range(22):  # Labels from 0 to 21
        label_data = df[df[label_column] == label]  # Filter rows for the label
        sampled_data = label_data.sample(frac=0.3, random_state=42)  # Randomly select 30%
        extracted_data.append(sampled_data)

    # Combine extracted data
    extracted_df = pd.concat(extracted_data)

    # Save to Excel
    extracted_df.to_excel(output_file, index=False)

    print(f"Extracted dataset saved to {output_file}")

# Example usage
input_file = "/content/drive/MyDrive/Colab-Notebooks/Dr.Saghafi/Assets/expanded_output_train.xlsx"  # Change to your actual file path
output_file = "/content/drive/MyDrive/Colab-Notebooks/Dr.Saghafi/Assets/expanded_output_test.xlsx"
extract_30_percent(input_file, output_file)

Extracted dataset saved to /content/drive/MyDrive/Colab-Notebooks/Dr.Saghafi/Assets/expanded_output_test.xlsx


In [3]:

# Preprocessing: Normalization and Data Splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Assuming the dataset is loaded into a DataFrame named 'data'
# Columns 1-9 are features, and column 10 is the label

# Splitting features and labels
train_data = pd.read_excel('/content/drive/MyDrive/Colab-Notebooks/Dr.Saghafi/Assets/expanded_output_train.xlsx')
X_train = train_data.iloc[:, :9].values
y_train = train_data.iloc[:, 9].values

test_data = pd.read_excel('/content/drive/MyDrive/Colab-Notebooks/Dr.Saghafi/Assets/expanded_output_test.xlsx')
x_test = test_data.iloc[:, :9].values
y_test = test_data.iloc[:, 9].values

# Normalizing features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

scaler = StandardScaler()
X_test = scaler.fit_transform(x_test)


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

def expand_dataset(input_file, output_file, expansion_factor=10000):
    # Read dataset
    # df = pd.read_excel(input_file)
    df = train_data

    # Extract features and category column
    feature_columns = df.columns[:-1]
    category_column = df.columns[-1]

    # Get unique categories
    categories = df[category_column].unique()

    new_samples = []

    for category in categories:
        # Filter dataset by category
        category_data = df[df[category_column] == category]

        # Store value distributions for each column
        column_distributions = {}

        for col in feature_columns:
            value_counts = Counter(category_data[col])
            most_common = value_counts.most_common(3)
            top_values, top_counts = zip(*most_common)

            total_top_counts = sum(top_counts)
            scaling_factor = 10000 / total_top_counts
            scaled_counts = {v: int(c * scaling_factor) for v, c in most_common}

            remaining_values = [v for v in value_counts.keys() if v not in top_values]
            remaining_counts = {v: value_counts[v] for v in remaining_values}

            column_distributions[col] = (scaled_counts, remaining_counts)

        # Generate new samples based on distributions
        for _ in range(expansion_factor):
            new_sample = {}
            for col in feature_columns:
                scaled_counts, remaining_counts = column_distributions[col]
                values = list(scaled_counts.keys()) + list(remaining_counts.keys())
                probabilities = [scaled_counts.get(v, 0) for v in values]
                probabilities = np.array(probabilities) / sum(probabilities)

                new_sample[col] = np.random.choice(values, p=probabilities)

            new_sample[category_column] = category
            new_samples.append(new_sample)

    # Convert to DataFrame and save to Excel
    expanded_df = pd.DataFrame(new_samples)
    expanded_df.to_excel(output_file, index=False)

# Example usage
expand_dataset("input.xlsx", "/content/drive/MyDrive/Colab-Notebooks/Dr.Saghafi/Assets/expanded_output_train.xlsx")

In [4]:

# Neural Network Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# One-hot encoding labels
num_classes = len(np.unique(y_train))
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# # Compute class weights to handle imbalance
# class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# class_weights = dict(enumerate(class_weights))

# # Model architecture
# model = Sequential([
#     Input(shape=(X_train.shape[1],)),  # Explicitly define the input shape here
#     Dense(128, activation='relu'),
#     BatchNormalization(),
#     Dropout(0.3),
#     Dense(64, activation='relu'),
#     BatchNormalization(),
#     Dropout(0.3),
#     Dense(num_classes, activation='softmax')
# ])

# # Compile the model
# model.compile(optimizer='AdamW', loss='categorical_crossentropy', metrics=['accuracy'])

# # Callbacks for early stopping and learning rate reduction
# early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-5)

# # Train the model
# history = model.fit(
#     X_train, y_train_cat,
#     validation_data=(X_train, y_train_cat),
#     epochs=400,
#     batch_size=32,
#     class_weight=class_weights
# )


In [5]:
!pip install geneticalgorithm
!pip install deap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for func-timeout: filename=func_timeout-4.3.5-py3-none-any.whl size=15076 sha256=6236854f91490fdd19be365006f5d7d43e088a0122b32e0f68329ab2cad676b7
  Stored in directory: /root/.cache/pip/wheels/07/e6/86/f23164d12c3134966614102db8e7956ab359faf7ffd78703ce
Successfully built func-timeout
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 3.0 MB/s eta 0:00:00


In [6]:
# from deap import base, creator, tools, algorithms
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
# import numpy as np
# import random

from deap import base, creator, tools, algorithms
import random
import numpy as np

# Hyperparameter ranges
BOUND_LOW = [32, 16, 0.1, 0.0001]  # Lower bounds: neurons1, neurons2, dropout, learning_rate
BOUND_UP = [256, 128, 0.5, 0.01]   # Upper bounds: neurons1, neurons2, dropout, learning_rate

# Evaluation function
def evaluate(params):
    # Ensure integer parameters for neurons
    neurons1 = int(params[0])
    neurons2 = int(params[1])
    dropout = params[2]
    learning_rate = params[3]

    if not (0 <= dropout <= 1):  # Validate dropout range
        return 0,  # Return a very low fitness if invalid

    try:
        # Placeholder for neural network training simulation
        accuracy = random.uniform(0, 1)  # Simulate accuracy
        return accuracy,
    except:
        return 0,  # If an error occurs, return a very low fitness

# DEAP Setup
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_float", lambda low, up: random.uniform(low, up))
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (lambda: toolbox.attr_float(BOUND_LOW[0], BOUND_UP[0]),
                  lambda: toolbox.attr_float(BOUND_LOW[1], BOUND_UP[1]),
                  lambda: toolbox.attr_float(BOUND_LOW[2], BOUND_UP[2]),
                  lambda: toolbox.attr_float(BOUND_LOW[3], BOUND_UP[3])), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)

# Custom mutation function for bounded integers and floats
def custom_mutation(individual, low, up):
    for i, val in enumerate(individual):
        if i < 2:  # For neurons (integers)
            individual[i] = random.randint(int(low[i]), int(up[i]))
        else:  # For dropout and learning rate (floats)
            individual[i] = random.uniform(low[i], up[i])
    return individual,

toolbox.register("mutate", custom_mutation, low=BOUND_LOW, up=BOUND_UP)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run GA
NGEN = 10  # Number of generations
POP_SIZE = 10  # Population size
pop = toolbox.population(n=POP_SIZE)

# Add statistics
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)

hof = tools.HallOfFame(1)

print("Running Genetic Algorithm...")
pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=NGEN,
                                stats=stats, halloffame=hof, verbose=True)

# Print best hyperparameters
best_params = hof[0]
best_neurons1 = int(best_params[0])
best_neurons2 = int(best_params[1])
best_dropout = best_params[2]
best_lr = best_params[3]

print(f"Best Parameters: Neurons1={best_neurons1}, Neurons2={best_neurons2}, Dropout={best_dropout}, Learning Rate={best_lr}")



# Build final model with best parameters
best_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(int(best_params[0]), activation='relu'),
    Dropout(best_params[2]),
    Dense(int(best_params[1]), activation='relu'),
    Dropout(best_params[2]),
    Dense(num_classes, activation='softmax')
])

# Compile and train the final model
best_model.compile(optimizer=Adam(learning_rate=best_params[3]),
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])
history = best_model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=50,
    batch_size=32,
    verbose=1
)


Running Genetic Algorithm...
gen	nevals	avg     	min     	max     
0  	10    	0.499501	0.272563	0.953554
1  	8     	0.618966	0.240318	0.97947 
2  	4     	0.631443	0.257871	0.926897
3  	8     	0.762147	0.198639	0.99682 
4  	7     	0.741861	0.0724723	0.99682 
5  	6     	0.677781	0.108665 	0.99682 
6  	5     	0.740401	0.390429 	0.99682 
7  	9     	0.544535	0.0954819	0.99682 
8  	6     	0.679096	0.109329 	0.99682 
9  	7     	0.602109	0.234187 	0.910913
10 	5     	0.578666	0.0199828	0.910913
Best Parameters: Neurons1=57, Neurons2=115, Dropout=0.09140077705799338, Learning Rate=0.005791365013379204
Epoch 1/50
6875/6875 ━━━━━━━━━━━━━━━━━━━━ 43s 6ms/step - accuracy: 0.5274 - loss: 1.3604 - val_accuracy: 0.7037 - val_loss: 0.7536
Epoch 2/50
6875/6875 ━━━━━━━━━━━━━━━━━━━━ 62s 3ms/step - accuracy: 0.6628 - loss: 0.8863 - val_accuracy: 0.7195 - val_loss: 0.7074
Epoch 3/50
6875/6875 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.6768 - loss: 0.8332 - val_accuracy: 0.7213 - val_loss: 0.6933
Epoch 4

In [ ]:

# Evaluate the model
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predict on test data
y_pred = best_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Classification report
print(classification_report(y_test, y_pred_classes))

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(X_test), yticklabels=np.unique(y_pred))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


2063/2063 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step
              precision    recall  f1-score   support

           0       0.68      0.73      0.71      3000
           1       0.77      0.70      0.74      3000
           2       0.57      0.61      0.59      3000
           3       0.83      0.71      0.77      3000
           4       0.66      0.71      0.68      3000
           5       0.64      0.72      0.68      3000
           6       0.84      0.85      0.85      3000
           7       0.64      0.73      0.68      3000
           8       0.69      0.86      0.76      3000
           9       0.75      0.75      0.75      3000
          10       0.83      0.92      0.87      3000
          11       0.76      0.63      0.69      3000
          12       0.54      0.74      0.62      3000
          13       0.80      0.53      0.64      3000
          14       0.68      0.68      0.68      3000
          15       0.58      0.46      0.51      3000
          16       0.77      0.59     